# 第19章　ハイパーパラメータ最適化と実験管理

**『医療診断支援AI開発　第2巻　応用編 ― 本格実装から臨床導入まで（第2巻　応用編）』のコード**

本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。

- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。
- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。
- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**

リポジトリ: https://github.com/kewel-corp/book-vol2

## 演習：探索を設計する

In [ ]:
import optuna

def train(lr, wd, aug, epochs, trial):
    # build_model / train_one_epoch / validate_ap は本書の最小構成（自分の学習コードに置き換える）
    model, opt = build_model(lr=lr, wd=wd)
    for epoch in range(epochs):
        train_one_epoch(model, opt, aug=aug)
        ap = validate_ap(model)                # 最大化したい指標と同じものを報告する
        trial.report(ap, step=epoch)           # ① 途中経過を Optuna に渡す
        if trial.should_prune():               # ② 打ち切るかどうかは pruner が判定する
            raise optuna.TrialPruned()         # ③ 例外を送出して試行を止める
    return model

def objective(trial):
    lr  = trial.suggest_float("lr", 1e-5, 1e-2, log=True)   # 対数スケール
    wd  = trial.suggest_float("wd", 1e-6, 1e-3, log=True)
    aug = trial.suggest_categorical("aug", ["weak", "strong"])
    model = train(lr=lr, wd=wd, aug=aug, epochs=15, trial=trial)
    if validate_fp_per_scan(model) > 2.0:  # 運用の必要条件を満たさない試行は不適格（学習途中の枝刈りとは別の理由）
        raise optuna.TrialPruned()
    return validate_ap(model)              # 検証データのAP（Average Precision、平均適合率）を最大化

study = optuna.create_study(direction="maximize",
                            pruner=optuna.pruners.MedianPruner())  # 中央値で枝刈り
study.optimize(objective, n_trials=40)
print(study.best_params)

## 相反する目標を同時に扱う：多目的最適化とPareto前線

In [ ]:
import optuna
def objective(trial):
    lr  = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    beta= trial.suggest_float("tversky_beta", 0.5, 0.85)
    model = train(lr=lr, beta=beta, epochs=15)
    sens  = validate_sensitivity(model)        # 最大化したい
    fp    = validate_fp_per_scan(model)         # 最小化したい
    return sens, fp

study = optuna.create_study(directions=["maximize", "minimize"],
                            sampler=optuna.samplers.NSGAIISampler())
study.optimize(objective, n_trials=60)
for t in study.best_trials:                     # Pareto前線上の解
    print(t.values, t.params)

## 平均とばらつきを用いて候補を比較する

In [ ]:
import numpy as np, optuna
from sklearn.model_selection import StratifiedGroupKFold

# X：症例の画像、y：正解マスク（Diceの評価用）、strata：層化用の1次元ラベル、
# patient_ids：患者ID（いずれも症例ごとに1要素の配列）
def cv_score(params, n_splits=5, seed=0):
    """一つのハイパーパラメータ組を、患者単位・層化のk-fold CVで評価する。"""
    skf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    scores = []
    for tr, va in skf.split(X, strata, groups=patient_ids):   # strata=層化用の1次元ラベル（例：腫瘍サイズ帯）、groups=患者ID（リーク防止）
        model = train(X[tr], y[tr], **params)
        scores.append(validate_dice(model, X[va], y[va]))
    return float(np.mean(scores)), float(np.std(scores))

def objective(trial):
    params = {
        "lr":   trial.suggest_float("lr", 1e-5, 1e-2, log=True),
        "wd":   trial.suggest_float("wd", 1e-6, 1e-2, log=True),
        "beta": trial.suggest_float("tversky_beta", 0.5, 0.85),
    }
    mean, std = cv_score(params)
    trial.set_user_attr("std", std)          # ばらつきも記録しておく
    return mean - 0.5 * std                  # 「平均が高く、foldで安定」を選ぶ

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)